# Eksperimen E5: Analisis Kasus Kegagalan (Failure Analysis)

Notebook ini mendemonstrasikan bagaimana kita mengaudit dan membedah kasus-kasus di mana AI (BirdNET/PANNs) **gagal** mengenali suara burung (*False Negatives*).
Fokus utama kita adalah mengklasifikasikan diagnosis kegagalan pada kondisi ekstrem (seperti Derau Kampus ITERA SNR -5 dB).

## Objektif:
1. Memuat tabel audit kegagalan nyata `failure_analysis_table.csv`.
2. Memvisualisasikan distribusi penyebab utama kegagalan (*Primary Cause*).
3. Menelusuri metadata spesifik kueri yang gagal untuk dilaporkan di Bab 4.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import IPython.display as ipd

# 1. Pastikan jalur repository sistem terbaca
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("[+] Pustaka E5 Berhasil Dimuat!")

## Tahap 1: Memuat Tabel Audit Kegagalan
Kita akan memuat 30 kasus representatif dari eksperimen retrieval yang mendapatkan nilai `top1_match == 0`.

In [ ]:
FAILURE_CSV = PROJECT_ROOT / "results/processed/failure_analysis_table.csv"

if FAILURE_CSV.exists():
    df_fail = pd.read_csv(FAILURE_CSV)
    print(f"[+] Ditemukan {len(df_fail)} kasus kegagalan.")
    display(df_fail.head(10))
else:
    print("[-] Tabel kegagalan belum ada. Jalankan src/analyze_failures.py terlebih dahulu.")

## Tahap 2: Visualisasi Distribusi Penyebab Kegagalan
Mengapa mesin bisa salah mengenali kicauan? Biasanya ada dua penyebab dominan di kondisi skripsi kita:
1. **Low SNR Masking:** Kicauan burung tertutup/kalah volume oleh bising kendaraan atau serangga.
2. **Acoustic Feature Overlap:** Kicauan burung sangat mirip dengan burung spesies lain di dalam database.

In [ ]:
if 'df_fail' in locals():
    plt.figure(figsize=(8, 6))
    sns.countplot(data=df_fail, y='primary_cause', palette='pastel', order=df_fail['primary_cause'].value_counts().index)
    
    plt.title("Distribusi Penyebab Kegagalan Retrieval (E5)", fontsize=14, pad=15)
    plt.xlabel("Jumlah Kasus", fontsize=12)
    plt.ylabel("Penyebab (Diagnosis)", fontsize=12)
    plt.tight_layout()
    plt.show()
    
    print("\n--- Rangkuman Persentase ---")
    print((df_fail['primary_cause'].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

## Tahap 3: Pemaparan Konteks Audio (Opsional untuk Bab 4)
Kita bisa mengambil 1 kueri yang gagal untuk didengarkan (jika audio mentahnya tersedia).

In [ ]:
if 'df_fail' in locals() and not df_fail.empty:
    sample_case = df_fail.iloc[0]
    print(f"CONTOH DIAGNOSIS KEGAGALAN: {sample_case['case_id']}")
    print(f"Spesies        : {sample_case['species_key']}")
    print(f"Model Represent: {sample_case['representation']}")
    print(f"Kondisi Derau  : {sample_case['condition']}")
    print(f"Skor Prediksi  : {sample_case['similarity_score']} (Dibawah Threshold {sample_case['threshold_tau']})")
    print(f"\n📝 Diagnosis Mesin:\n{sample_case['diagnosis_and_threat']}")